In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

In [ ]:
Fight_db = pd.read_csv('ufc_fight_data.csv')
Fight_db = Fight_db.sort_values(by= 'fight_id', ascending=False)


Fight_db = Fight_db.dropna(subset=['f_1'])
Fight_db = Fight_db.dropna(subset=['f_2'])
Fight_db['weight_class'] = Fight_db['weight_class'].fillna('Catch Weight')

Fight_db = Fight_db[Fight_db['event_id'] >= 142]
Fight_db = Fight_db[Fight_db['gender'] == 'M']
Fight_db = Fight_db.drop(['referee','gender',	'result',	'result_details', 'finish_time', 'weight_class', 'finish_round',	'fight_url'], axis=1)
Fight_db['title_fight'].replace(['F','T'],[0, 1],inplace=True)
Fight_db

,fight_id,event_id,f_1,f_2,winner,num_rounds,title_fight
0,7218,664,2976.0,2884.0,2884.0,5,0
1,7217,664,1662.0,2464.0,1662.0,3,0
3,7215,664,3831.0,2974.0,3831.0,3,0
4,7214,664,1108.0,2320.0,2320.0,3,0
5,7213,664,3945.0,2373.0,2373.0,3,0
...,...,...,...,...,...,...,...
5963,1255,142,2293.0,226.0,2293.0,3,0
5964,1254,142,1692.0,2101.0,1692.0,3,0
5965,1253,142,1569.0,2388.0,1569.0,3,0
5966,1252,142,3102.0,1143.0,3102.0,3,0


In [ ]:
def determinar_ganador(row):
    if row['winner'] == float(row['f_1']):
        return -1
    elif row['winner'] == float(row['f_2']):
        return 1
    else:
        # Este caso maneja situaciones donde 'winner' no es ni 'f_1' ni 'f_2'
        # Podría indicar un error o una situación no prevista en los datos
        return 0

In [ ]:
Fight_db['winner'] = Fight_db.apply(determinar_ganador, axis=1)
Fight_db

fighter_ids = set(Fight_db['f_1']).union(set(Fight_db['f_2']))

Fighter_db = pd.read_csv('ufc_fighter_data.csv')
Fighter_db = Fighter_db.sort_values(by= 'fighter_id', ascending=False)
Fighter_db

Fighter_db = Fighter_db.drop(['fighter_f_name', 'fighter_l_name','fighter_nickname', 'fighter_stance',	'fighter_d', 'fighter_nc_dq',	'fighter_url'], axis=1)
Fighter_db = Fighter_db[Fighter_db['fighter_id'].isin(fighter_ids)]
Fighter_db

Fighter_db.isnull().sum()

mean_value = Fighter_db['fighter_height_cm'].mean()
Fighter_db['fighter_height_cm'].fillna(mean_value, inplace=True)
mean_value = Fighter_db['fighter_reach_cm'].mean()
Fighter_db['fighter_reach_cm'].fillna(mean_value, inplace=True)
Fighter_db['fighter_dob'] = Fighter_db['fighter_dob'].fillna('1992-12-24')

Event_db = pd.read_csv('ufc_event_data.csv')

<ipython-input-6-7f73d517db15>:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Fighter_db['fighter_height_cm'].fillna(mean_value, inplace=True)
<ipython-input-6-7f73d517db15>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Fighter_db['fighter_reach_cm'].fillna(mean_value, inplace=True)
<ipython-input-6-7f73d517db15>:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Fighter_db

In [ ]:
def calculate_age(event_date, date_of_birth):
    # Ensure that both event_date and date_of_birth are converted to datetime objects if not already
    # event_date = pd.to_datetime(event_date)
    # date_of_birth = pd.to_datetime(date_of_birth)

    # Calculate age as the difference in years between the event date and the date of birth
    age = event_date.year - date_of_birth.year

    # Adjust for cases where the person's birthday hasn't occurred yet in the given year
    if (event_date.month, event_date.day) < (date_of_birth.month, date_of_birth.day):
        age -= 1

    return age

def add_fighters_ages_to_fights(Fight_db, Fighter_db, Event_db):
    Event_db['event_date'] = pd.to_datetime(Event_db['event_date'])
    Fighter_db['fighter_dob'] = pd.to_datetime(Fighter_db['fighter_dob'])

    # Add columns to store ages in fight_data
    Fight_db['fighter_1_age'] = pd.NA
    Fight_db['fighter_2_age'] = pd.NA

    for index, fight in Fight_db.iterrows():
        # Get the event_date for the current fight
        matched_events = Event_db.loc[Event_db['event_id'] == fight['event_id'], 'event_date']
        if not matched_events.empty:
            event_date = matched_events.iloc[0]
        else:
            print(f"No matching event found for event_id: {fight['event_id']}")
            continue  # Skip to the next iteration if no match is found

        # Get fighter 1's date_of_birth
        fighter_1_dob = Fighter_db.loc[Fighter_db['fighter_id'] == fight['f_1'], 'fighter_dob'].iloc[0]

        # Get fighter 2's date_of_birth
        fighter_2_dob = Fighter_db.loc[Fighter_db['fighter_id'] == fight['f_2'], 'fighter_dob'].iloc[0]

        # Calculate each fighter's age and store it in the fight_data DataFrame
        Fight_db.at[index, 'fighter_1_age'] = calculate_age(event_date, fighter_1_dob)
        Fight_db.at[index, 'fighter_2_age'] = calculate_age(event_date, fighter_2_dob)

    return Fight_db


    print(Fight_db.columns)  # This will show all column names in Fight_db
    Fight_db_with_ages =add_fighters_ages_to_fights(Fight_db, Fighter_db, Event_db)

In [ ]:
FightStat_db = pd.read_csv('ufc_fight_stat_data.csv')
FightStat_db = FightStat_db.sort_values(by= 'fight_stat_id', ascending=False)
FightStat_db

FightStat_db = FightStat_db.drop(['fight_stat_id', 'reversals','fight_url'], axis=1)
FightStat_db = FightStat_db[FightStat_db['fighter_id'].isin(fighter_ids)]
FightStat_db = FightStat_db[FightStat_db['fight_id'] >= 1251]
FightStat_db

,fight_id,fighter_id,knockdowns,total_strikes_att,total_strikes_succ,sig_strikes_att,sig_strikes_succ,takedown_att,takedown_succ,submission_att,ctrl_time
0,7218,2976.0,0.0,34.0,19.0,32.0,18.0,0.0,0.0,0.0,0:00
1,7218,2884.0,0.0,42.0,17.0,40.0,16.0,6.0,1.0,0.0,1:28
2,7217,1662.0,0.0,59.0,37.0,40.0,23.0,15.0,5.0,1.0,7:33
3,7217,2464.0,0.0,72.0,32.0,55.0,18.0,0.0,0.0,0.0,2:11
6,7215,3831.0,0.0,105.0,63.0,80.0,45.0,2.0,1.0,1.0,3:30
...,...,...,...,...,...,...,...,...,...,...,...
11931,1253,2388.0,0.0,17.0,10.0,16.0,9.0,2.0,0.0,1.0,0:21
11932,1252,3102.0,1.0,56.0,30.0,39.0,15.0,2.0,2.0,0.0,4:38
11933,1252,1143.0,0.0,34.0,29.0,8.0,5.0,1.0,0.0,1.0,0:00
11934,1251,1402.0,0.0,83.0,55.0,45.0,21.0,7.0,6.0,4.0,11:22


In [ ]:
def calculate_previous_8_fight_average(FightStat_db, fighter_id, fight_id):
    # Filtrar el DataFrame para el fighter_id dado y excluir el fight_id actual
    fighter_history = FightStat_db[(FightStat_db['fighter_id'] == fighter_id) & (FightStat_db['fight_id'] != fight_id)]

    # Ordenar los datos por fight_id en orden descendente para obtener las peleas anteriores
    fighter_history = fighter_history.sort_values(by='fight_id', ascending=False)

    # Tomar las últimas 8 peleas o menos si no hay suficientes registros
    latest_8_or_less_fights = fighter_history.head(8)

    # Calcular el promedio excluyendo columnas no deseadas
    columns_to_exclude = [ 'fight_id', 'fighter_id']
    average_stats = latest_8_or_less_fights.drop(columns=columns_to_exclude).mean(numeric_only=True)

    # Añadir el prefijo 'mean_' a las columnas del promedio
    average_stats = average_stats.add_prefix('mean_')

    return average_stats

def add_mean_stats_to_fights(df):
    # Aplicar la función a cada fila
    mean_stats_list = []
    for index, row in df.iterrows():
        mean_stats = calculate_previous_8_fight_average(df, row['fighter_id'], row['fight_id'])
        mean_stats_list.append(mean_stats)

    # Crear un DataFrame con los promedios calculados
    mean_stats_df = pd.DataFrame(mean_stats_list)
    mean_stats_df.index = df.index  # Asegurar que los índices coincidan para poder unirlos

    # Unir el DataFrame original con las estadísticas promedio calculadas
    updated_df = pd.concat([df, mean_stats_df], axis=1)

    return updated_df

In [ ]:
# Suponiendo que ya tienes un DataFrame llamado df que representa los datos cargados del archivo CSV.
# Ejemplo de cómo aplicar estas funciones:
updated_df = add_mean_stats_to_fights(FightStat_db)


new_fighter_ids = [1426, 263]  # List of fighter IDs for the new fight
new_fight_id = 7220  # ID for the new fight (replace 12345 with the actual fight ID)

# Calculate average statistics for each fighter in the new fight
example_means = {}
for fighter_id in new_fighter_ids:
    example_means[fighter_id] = calculate_previous_8_fight_average(FightStat_db, fighter_id, new_fight_id)

# Print example means for each fighter in the new fight
for fighter_id, mean_stats in example_means.items():
    print(f"Example means for Fighter {fighter_id}:")
    print(mean_stats)
    print()

updated_df = updated_df.fillna(0)

updated_df.to_csv('Mean_Ufc_db.csv', index=False)

updated_df = updated_df.drop(['fighter_id',
    'knockdowns',
    'total_strikes_att',
    'total_strikes_succ',
    'sig_strikes_att',
    'sig_strikes_succ',
    'takedown_att',
    'takedown_succ',
    'submission_att',
    'ctrl_time'], axis=1)


df_even = updated_df[updated_df.index % 2 == 0]
df_even = df_even.add_prefix('f1_')
df_odd = updated_df[updated_df.index % 2 != 0]
df_odd = df_odd.add_prefix('f2_')
mean_stats_df = pd.merge(df_even, df_odd, left_on= 'f1_fight_id', right_on='f2_fight_id', how='inner')
mean_stats_df = mean_stats_df.drop(['f2_fight_id'], axis=1)
mean_stats_df
#mean_stats_df = mean_stats_df.dropna()
#mean_stats_df

Fighter_db_f1= Fighter_db
Fighter_db_f1 = Fighter_db_f1.add_prefix('f1_')
Fighter_db_f2= Fighter_db
Fighter_db_f2 = Fighter_db_f2.add_prefix('f2_')

Ufc_db = pd.merge(updated_df, Fighter_db_f1, left_on= 'f_1', right_on='f1_fighter_id', how='left')
#Ufc_db = Ufc_db.drop(['f1_fighter_id'], axis=1)
Ufc_db = pd.merge(Ufc_db, Fighter_db_f2, left_on= 'f_2', right_on='f2_fighter_id', how='left')
#Ufc_db = Ufc_db.drop(['f2_fighter_id'], axis=1)
Ufc_db = pd.merge(Ufc_db, mean_stats_df, left_on= 'fight_id', right_on='f1_fight_id', how='left')
Ufc_db = Ufc_db.drop(['f1_fight_id'], axis=1)
Ufc_db

Example means for Fighter 1426:
mean_knockdowns              0.750
mean_total_strikes_att     101.000
mean_total_strikes_succ     64.875
mean_sig_strikes_att        98.750
mean_sig_strikes_succ       63.125
mean_takedown_att            0.125
mean_takedown_succ           0.000
mean_submission_att          0.000
dtype: float64

Example means for Fighter 263:
mean_knockdowns              0.875
mean_total_strikes_att     175.125
mean_total_strikes_succ     97.000
mean_sig_strikes_att       160.750
mean_sig_strikes_succ       84.500
mean_takedown_att            0.875
mean_takedown_succ           0.250
mean_submission_att          0.625
dtype: float64



KeyError: 'f_1'

In [ ]:
desired_columns = ['fight_id','f1_fighter_id','f2_fighter_id',
                   'f1_fighter_height_cm',	'f1_fighter_weight_lbs',	'f1_fighter_reach_cm',	'fighter_1_age',	'f1_fighter_w',	'f1_fighter_l',
                   'f1_mean_knockdowns',	'f1_mean_total_strikes_att',	'f1_mean_total_strikes_succ',	'f1_mean_sig_strikes_att',	'f1_mean_sig_strikes_succ',	'f1_mean_takedown_att',	'f1_mean_takedown_succ',	'f1_mean_submission_att',
                   'f2_fighter_height_cm',	'f2_fighter_weight_lbs',	'f2_fighter_reach_cm',	'fighter_2_age',	'f2_fighter_w',	'f2_fighter_l',
                   'f2_mean_knockdowns',	'f2_mean_total_strikes_att',	'f2_mean_total_strikes_succ',	'f2_mean_sig_strikes_att',	'f2_mean_sig_strikes_succ',	'f2_mean_takedown_att',	'f2_mean_takedown_succ',	'f2_mean_submission_att',
                   'num_rounds',	'title_fight',
                   'winner']
#print('column names of Ufc_db>',Ufc_db.columns)

Final_Ufc_db = Ufc_db[desired_columns]

Final_Ufc_db.isnull().sum()

Final_Ufc_db.to_csv('Final_Ufc_db.csv', index=False)